# Solar Filament Segmentation Challenge 2026 — Reproduction notebook

End-to-end: **weights → submission.csv** (no training required).

Set the paths in the first cell, then *Run All*. Outputs `submission.csv` (official
schema) and `audit.json` (schema / positive-area / disjointness checks).

In [ ]:
from pathlib import Path
import json, sys, shutil

def _find(name):
    hits = [p for p in Path('/kaggle/input').rglob(name) if p.is_file()]
    assert hits, f'missing {name}'
    return hits[0]

# --- configure (Kaggle defaults; override for local runs) -------------------
WEIGHTS     = _find('alldata_convnext_32768.pt')   # flagship; alternatives: alldata_convnext_16384.pt, convnext_long_fold1_16384.pt
TEST_IMAGES = next(d for d in Path('/kaggle/input').rglob('test_images')
                   if d.is_dir() and len(list(d.glob('*.jpeg'))) == 180)
OUTPUT      = Path('/kaggle/working')

SRC_DIR = Path('/kaggle/working/src')
if not (SRC_DIR / 'solar_filament').exists():
    pkg = next(p for p in Path('/kaggle/input').rglob('artifacts.py')
               if p.parent.name == 'solar_filament')
    SRC_DIR.mkdir(exist_ok=True)
    shutil.copytree(pkg.parent, SRC_DIR / 'solar_filament')
sys.path.insert(0, str(SRC_DIR))
print('weights:', WEIGHTS)
print('test:', TEST_IMAGES)

In [ ]:
# --- dependencies (pinned) --------------------------------------------------
import importlib.metadata as im, subprocess
def ensure(pkg, version):
    try:
        if im.version(pkg) == version:
            return
    except im.PackageNotFoundError:
        pass
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{pkg}=={version}'], check=True)
ensure('segmentation-models-pytorch', '0.5.0')

import hashlib, time
import numpy as np
import pandas as pd
import torch
from PIL import Image
from pycocotools import mask as mutils
from scipy import ndimage
from solar_filament.config import TrainingConfig
from solar_filament.masks import encode_mask
from solar_filament.network import build_network
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# --- load checkpoint, verify against the pinned manifest ---------------------
h = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()
print('weight sha256:', h)
manifest_path = SRC_DIR.parent / 'weights_manifest.json'
if not manifest_path.exists():
    manifest_path = Path('weights_manifest.json')
if manifest_path.exists():
    known = json.loads(manifest_path.read_text())
    assert h in known.values(), 'checkpoint hash is not pinned in weights_manifest.json'
else:
    print('WARNING: no weights_manifest.json found; hash recorded only')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CONFIG = TrainingConfig(
    encoder='tu-convnext_tiny.fb_in1k', encoder_weights='imagenet',
    crop_size=512, batch_size=2, learning_rate=2e-4, weight_decay=1e-4,
    max_steps=16384, seed=2026, validation_fold=1,
    device='cuda' if device.type == 'cuda' else 'cpu',
)
network = build_network(CONFIG, initialize_pretrained=False)  # full encoder state ships in the checkpoint
state = torch.load(WEIGHTS, map_location='cpu', weights_only=False)
network.load_state_dict(state['model_state'], strict=True)
network.to(device).eval()
print('loaded checkpoint at step', state.get('step'))

In [ ]:
# --- 4-rotation TTA inference at native resolution ---------------------------
def normalize_grayscale(p):
    assert p.ndim == 2 and p.dtype == np.uint8
    return (p.astype(np.float32) / 255.0 - 0.5) / 0.25

def tile_positions(length, tile_size, stride):
    positions = list(range(0, max(length - tile_size, 0) + 1, stride))
    last = max(length - tile_size, 0)
    if positions[-1] != last:
        positions.append(last)
    return positions

def tiled(network, pixels, tile_size=512, batch_size=2, device=device):
    stride = tile_size // 2
    normalized = normalize_grayscale(pixels)
    h, w = pixels.shape
    padded = np.pad(normalized, ((0, max(0, tile_size - h)), (0, max(0, tile_size - w))),
                    constant_values=-2.0)
    positions = [(y, x) for y in tile_positions(padded.shape[0], tile_size, stride)
                       for x in tile_positions(padded.shape[1], tile_size, stride)]
    window = np.outer(np.hanning(tile_size), np.hanning(tile_size)).astype(np.float32)
    window = np.maximum(window, 0.01)
    weighted = np.zeros((1, *padded.shape), dtype=np.float32)
    weights = np.zeros(padded.shape, dtype=np.float32)
    for s in range(0, len(positions), batch_size):
        batch = positions[s:s + batch_size]
        tiles = np.stack([padded[y:y + tile_size, x:x + tile_size] for y, x in batch])[:, None]
        tensor = torch.from_numpy(tiles).to(device)
        with torch.inference_mode(), torch.autocast(device_type=device.type,
                                                    enabled=device.type == 'cuda'):
            probs = network(tensor).float().sigmoid().cpu().numpy()
        assert np.isfinite(probs).all()
        for (y, x), p in zip(batch, probs):
            weighted[:, y:y + tile_size, x:x + tile_size] += p * window
            weights[y:y + tile_size, x:x + tile_size] += window
    return np.clip((weighted[:, :h, :w] / weights[None, :h, :w])[0], 0, 1)

def rotation_averaged(network, pixels, device=device):
    identity = tiled(network, pixels, device=device)
    acc = identity.copy()
    for turns in (1, 2, 3):
        rotated = np.ascontiguousarray(np.rot90(pixels, turns))
        aligned = np.rot90(tiled(network, rotated, device=device), -turns, axes=(-2, -1))
        assert aligned.shape == identity.shape
        acc += aligned
    return (acc * np.float32(0.25)).astype(np.float32)

def extract_instances(prob, threshold=0.5, min_area=32):
    labels, count = ndimage.label(prob >= threshold, structure=np.ones((3, 3)))
    if not count:
        return []
    areas = np.bincount(labels.ravel())
    keep = np.flatnonzero(areas[1:] >= min_area) + 1
    out = []
    for label in keep:
        binary = labels == label
        encoded = encode_mask(binary)
        out.append({**encoded, 'area': int(binary.sum()), 'confidence': float(prob[binary].mean())})
    return sorted(out, key=lambda i: -i['confidence'])

rows, started = [], time.perf_counter()
names = sorted(p.name for p in TEST_IMAGES.glob('*.jpeg'))
assert len(names) == 180, names
for index, filename in enumerate(names, 1):
    with Image.open(TEST_IMAGES / filename) as image:
        pixels = np.asarray(image)
    prob = rotation_averaged(network, pixels)
    for inst in extract_instances(prob, 0.55, 32):
        if inst['area'] >= 128 and inst['confidence'] >= 0.95:  # sweep-winning filter (fold1 +0.0056)
            rows.append({'photo': filename.rsplit('.', 1)[0], 'rle': inst['counts']})
    if index == 1 or index % 20 == 0 or index == len(names):
        print(f'{index}/180 elapsed={time.perf_counter() - started:.0f}s', flush=True)

In [ ]:
# --- submission + audit ------------------------------------------------------
counter, final = {}, []
for row in rows:
    counter[row['photo']] = counter.get(row['photo'], 0) + 1
    final.append({'filament_id': f"{row['photo']}_{counter[row['photo']]}",
                  'segmentation_rle': row['rle']})
frame = pd.DataFrame(final, columns=['filament_id', 'segmentation_rle'])
frame.to_csv(OUTPUT / 'submission.csv', index=False)

audit = {'rows': len(frame), 'unique_ids': int(frame['filament_id'].nunique()),
         'photos': int(frame['filament_id'].str.rsplit('_', n=1).str[0].nunique()),
         'schema_ok': list(frame.columns) == ['filament_id', 'segmentation_rle'],
         'positive_area': True, 'disjoint': True}
occupied = {}
for _, r in frame.iterrows():
    photo = r['filament_id'].rsplit('_', 1)[0]
    mask = mutils.decode({'size': [2048, 2048],
                          'counts': r['segmentation_rle']}).astype(bool)
    audit['positive_area'] &= bool(mask.any())
    if photo in occupied:
        audit['disjoint'] &= not bool((mask & occupied[photo]).any())
        occupied[photo] |= mask
    else:
        occupied[photo] = mask
(OUTPUT / 'audit.json').write_text(json.dumps(audit, indent=1))
print(audit)